# House Price Prediction

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

## 1. Load Dataset

In [ ]:
df = pd.read_csv('../dataset/houses.csv')
df.head()

## 2. Exploratory Data Analysis & Preprocessing

In [ ]:
df.info()

In [ ]:
# Handle missing values and extract numeric size
df = df.dropna(subset=['location', 'size', 'bath', 'total_sqft'])
df['bedrooms'] = df['size'].apply(lambda x: int(x.split(' ')[0]))

def convert_sqft_to_num(x):
    tokens = str(x).split('-')
    if len(tokens) == 2:
        return (float(tokens[0]) + float(tokens[1])) / 2
    try:
        return float(x)
    except:
        return np.nan
        
df['area'] = df['total_sqft'].apply(convert_sqft_to_num)
df = df.dropna(subset=['area'])

In [ ]:
# Rename columns
df = df.rename(columns={'bath': 'bathrooms'})
df['price'] = df['price'] * 100000  # Convert Lakhs to Rupees

# Keep relevant columns
df = df[['area_type', 'location', 'area', 'bedrooms', 'bathrooms', 'balcony', 'price']]
df.head()

## 3. Data Visualization

In [ ]:
plt.figure(figsize=(10,6))
sns.histplot(df['price'], bins=50)
plt.title('Distribution of House Prices')
plt.show()

In [ ]:
plt.figure(figsize=(10,6))
sns.scatterplot(x='area', y='price', data=df)
plt.title('Area vs Price')
plt.show()

## 4. Model Training

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score

X = df.drop('price', axis=1)
y = df['price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
num_cols = ['area', 'bedrooms', 'bathrooms', 'balcony']
cat_cols = ['area_type', 'location']

num_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])

model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(n_estimators=50, random_state=42))
])

model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print("R2 Score:", r2_score(y_test, y_pred))